In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'

donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', index_col=0)
ips_clusters.shape

In [ ]:
ips_clusters.head()

In [ ]:
adata_path = f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad'
adata = sc.read_h5ad(adata_path)

In [ ]:
adata

In [ ]:
adata.obs.index = [x.replace('-0', '').replace('-1', '') for x in adata.obs.index]

In [ ]:
adata.obs['ips_subtype'] = ips_clusters.reindex(adata.obs.index)['ips_subtype'].fillna('other')

In [ ]:
adata.obs.head()

### first do merged together

In [ ]:
n_neighbors = 50
sc.pp.neighbors(adata, n_neighbors=n_neighbors, random_state=0, use_rep='X_pca_harmony')

In [ ]:
def compute_prop_same(i):
    cell_cluster = groups[i]
    neighbors = distances[:, i].nonzero()[0]
    if len(neighbors) == 0:
        return np.nan
    total_same = np.sum(groups[neighbors] == cell_cluster)
    return total_same / len(neighbors)

In [ ]:
tmp_meta = adata.obs.copy()
# )

In [ ]:
###
distances = adata.obsp['distances']
cells = np.arange(adata.shape[0])

### ips_subtype
groups = adata.obs['line'].values
with ProcessPoolExecutor() as executor:
    ips_subtype_results = list(tqdm(executor.map(compute_prop_same, cells), total=len(cells)))
tmp_meta['line_prop'] = ips_subtype_results

In [ ]:
x_lab = 'ips_subtype'
y_lab = 'ips_subtype_prop'
hue = 'line'

fig, axes = plt.subplots(1, figsize=(6, 5))

ax = sns.lineplot(data=tmp_meta, x=x_lab, y=y_lab, hue=hue)

ax.set_xlabel('', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelsize=14, labelrotation=45)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

### now test ips subtype

In [ ]:
adata_path = f'{projdir}/h5ad/chrom100k/ips_subtype.processed.h5ad'
ips_adata = sc.read_h5ad(adata_path)

In [ ]:
ips_adata

In [ ]:
sc.pl.umap(ips_adata, color=['ips_subtype', 'line'])

In [ ]:
n_neighbors = 50
sc.pp.neighbors(ips_adata, n_neighbors=n_neighbors, random_state=0, use_rep='X_pca')

In [ ]:
ips_meta = ips_adata.obs.copy()

In [ ]:
###
distances = ips_adata.obsp['distances']
cells = np.arange(ips_adata.shape[0])

### ips_subtype
groups = ips_adata.obs['line'].values
with ProcessPoolExecutor() as executor:
    ips_subtype_results = list(tqdm(executor.map(compute_prop_same, cells), total=len(cells)))
ips_meta['donor_prop'] = ips_subtype_results

In [ ]:
ips_meta['ips_subtype'].value_counts()

In [ ]:
x_lab = 'ips_subtype'
y_lab = 'donor_prop'
hue = 'line'

fig, axes = plt.subplots(1, figsize=(6, 5))

ax = sns.lineplot(data=ips_meta, x=x_lab, y=y_lab, hue=hue)

ax.set_xlabel('', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelsize=14, labelrotation=45)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
ips_meta.to_csv(f'{outdir}/ips_subtype_nn_prop.csv', sep='\t')

In [ ]:
!date